In [ ]:
%%capture
%pip install openai pydantic pandas pyarrow numpy

In [ ]:
from openai import OpenAI
import os
import pandas as pd
import json
from pydantic import BaseModel, Field
from typing import List, Literal, Dict, Callable, Optional, Any
from enum import Enum
import numpy as np
import time

In [ ]:
# ==================== CONFIGURATION ====================
# Paths
BATCH_EVAL_DIR = "../data/cache/batch_json/"
CACHE_SAVE_FOLDER = "../data/cache/eval-cache/"
os.makedirs(CACHE_SAVE_FOLDER, exist_ok=True)
os.makedirs(BATCH_EVAL_DIR, exist_ok=True)

BATCH_EVAL_PATH = os.path.join(BATCH_EVAL_DIR, "batch_evaluation_requests_{}.jsonl")

INPUT_PATH = "../data/processed/Thuan_QA_data.parquet"
EVAL_OUTPUT_PATH_PARQUET = "../data/processed/Thuan_QA_data_evaluated.parquet"

# Cache paths (incremental saving/loading)
CACHE_LOAD_FOLDER = "../data/cache/eval-cache/"

# Resume configuration
RESUME_FROM_LAST_CHECKPOINT = False

# Model settings
MODEL_NAME = "gpt-4.1-mini"
GPT_API_KEY = os.getenv("OPENAI_API_KEY")

# Batch processing
BATCH_CHUNK_SIZE = 1500  # requests per batch file (max 50,000 per OpenAI)
SAMPLE_SIZE = 20000
RANDOM_STATE = 42

# ==================== PYDANTIC MODELS ====================

class CriterionScore(BaseModel):
    score: Literal[0, 1, 2] = Field(..., description="Điểm cho tiêu chí này")
    reason: str = Field(..., description="Lý do ngắn gọn (tiếng Việt, tối đa 15 từ)")

class OverallAssessment(str, Enum):
    ACCEPT = "ACCEPT"
    REJECT = "REJECT"
    REVISE = "REVISE"

class QAEvaluationOutput(BaseModel):
    answerability: CriterionScore
    clarity: CriterionScore
    answer_accuracy: CriterionScore
    conciseness: CriterionScore
    usefulness: CriterionScore
    total_score: int = Field(..., description="Tổng điểm các tiêu chí (0-10)")
    overall_assessment: OverallAssessment
    overall_reason: str = Field(..., description="Tổng kết ngắn gọn (tiếng Việt, tối đa 25 từ)")

    def model_post_init(self, __context):
        """Validate total score matches sum of criteria"""
        calculated_total = (
            self.answerability.score +
            self.clarity.score +
            self.answer_accuracy.score +
            self.conciseness.score +
            self.usefulness.score
        )
        if self.total_score != calculated_total:
            raise ValueError(f"Total score {self.total_score} != sum of criteria {calculated_total}")

def make_openai_strict_schema(pydantic_model) -> dict:
    """
    Convert a Pydantic model's JSON schema into one compatible with
    OpenAI Structured Outputs (strict mode).
    
    Fixes applied:
      - Adds "additionalProperties": false to every object
      - Ensures all object properties are listed in "required"
      - Strips unsupported keywords (title, default, minLength, maxLength, etc.)
    """
    schema = pydantic_model.model_json_schema()
    
    UNSUPPORTED_KEYS = {"title", "default", "minLength", "maxLength", "minimum", "maximum",
                        "exclusiveMinimum", "exclusiveMaximum", "multipleOf", "pattern", "format"}
    
    def _fix_object(obj):
        if not isinstance(obj, dict):
            return obj
        
        # Strip unsupported keys
        for key in UNSUPPORTED_KEYS:
            obj.pop(key, None)
        
        # If it's an object type, enforce strict rules
        if obj.get("type") == "object" and "properties" in obj:
            obj["additionalProperties"] = False
            obj["required"] = list(obj["properties"].keys())
            for prop in obj["properties"].values():
                _fix_object(prop)
        
        # Recurse into arrays
        if "items" in obj:
            _fix_object(obj["items"])
        
        # Recurse into anyOf / oneOf / allOf
        for combo_key in ("anyOf", "oneOf", "allOf"):
            if combo_key in obj:
                for sub in obj[combo_key]:
                    _fix_object(sub)
        
        return obj
    
    # Fix all $defs first
    if "$defs" in schema:
        for def_name, def_schema in schema["$defs"].items():
            _fix_object(def_schema)
    
    # Fix root
    _fix_object(schema)
    
    return schema

# ==================== PROMPTS ====================
# Cost optimization: static instructions go in the system message (cached by OpenAI
# across requests in the same batch), dynamic per-QA data goes in the user message.

SYSTEM_PROMPT = """You are a strict QA evaluator for Vietnamese financial news.
Evaluate each QA pair using 5 criteria (0-2 points each, total 0-10).
**ALL evaluation reasons MUST be written in Vietnamese. Max 15 words per reason, 25 words for overall_reason.**

## CRITERIA (0-2 points)

### 1. ANSWERABILITY - Is info in text?
- 0: Cannot answer from text / needs external knowledge
- 1: Answerable but needs inference / combining sentences
- 2: Direct answer, clear info in text

### 2. CLARITY - Is question clear?
- 0: Vague, missing subject/object
- 1: Understandable but lacks context (time, specific entity)
- 2: Clear, specific, natural

### 3. ANSWER_ACCURACY - Is answer correct?
- 0: Wrong info / numbers
- 1: Correct but missing important info
- 2: Accurate, complete, matches text

### 4. CONCISENESS - Is answer concise?
- 0: Verbose, many redundant words
- 1: Slightly long, could be shorter
- 2: Brief, appropriate length

### 5. USEFULNESS - Is question useful?
- 0: Meaningless / too trivial
- 1: Secondary info, less important
- 2: Core info users need

## THRESHOLDS
- **ACCEPT:** Total ≥ 8 AND answer_accuracy = 2 AND no criterion = 0
- **REVISE:** Total 5-7 OR 1 criterion = 0 OR answer_accuracy = 1
- **REJECT:** Total < 5 OR ≥2 criteria = 0 OR answer_accuracy = 0

## RULES
- When in doubt, score LOW
- Check numbers carefully - most error-prone
- answer_accuracy is most important"""

# Compact user message — only dynamic data, no repeated instructions
USER_PROMPT_TEMPLATE = """Title: {title}
Date: {timestamp}
Context: {context}

Question: {question}
Answer: {answer}"""

# ==================== CLIENT ====================
client = OpenAI(api_key=GPT_API_KEY)

In [ ]:
def clear_batches():
    """Cancel all in-progress batches."""
    all_batches = list(client.batches.list(limit=100))
    print(f"Found {len(all_batches)} batches")
    num_cancelled = 0

    for batch in all_batches:
        if batch.status in ("validating", "in_progress", "finalizing"):
            try:
                client.batches.cancel(batch.id)
                num_cancelled += 1
            except Exception as e:
                print(f"Skipping {batch.id}: {e}")

    if num_cancelled > 0:
        print(f"Cancelled {num_cancelled} batches")
    else:
        print("No active batches to cancel")

# clear_batches()

# Data Loading

In [ ]:
df = pd.read_parquet(INPUT_PATH, engine="pyarrow")
# df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

# Count QA pairs before cleaning
qa_count_before = 0
for _, row in df.iterrows():
    qa_pairs = row.get('qa_pairs', [])
    if isinstance(qa_pairs, np.ndarray):
        qa_count_before += sum(1 for qa in qa_pairs if isinstance(qa, dict))

# Remove empty QA pairs (questions without answers)
for idx, row in df.iterrows():
    qa_pairs = row.get('qa_pairs', [])
    if isinstance(qa_pairs, np.ndarray):
        row['qa_pairs'] = [
            qa for qa in qa_pairs
            if isinstance(qa, dict) and qa.get('question', '').strip() and qa.get('answer', '').strip()
        ]

# Count QA pairs after cleaning
qa_count_after = 0
for _, row in df.iterrows():
    qa_pairs = row.get('qa_pairs', [])
    if isinstance(qa_pairs, list):
        qa_count_after += sum(1 for qa in qa_pairs if isinstance(qa, dict))

removed_count = qa_count_before - qa_count_after

print(f"[INFO] Loaded {len(df)} articles from {INPUT_PATH}")
print(f"[INFO] QA pairs before cleaning: {qa_count_before}")
print(f"[INFO] QA pairs after cleaning: {qa_count_after}")
print(f"[INFO] Removed {removed_count} empty QA pairs ({removed_count/qa_count_before*100:.1f}%)" if qa_count_before > 0 else "[INFO] No QA pairs to clean")

# Batch Processing Utilities

Reusable functions for batch file creation, job submission, and result parsing.

In [ ]:
# ==================== BATCH PROCESSING UTILITIES (OpenAI Batch API) ====================

def upload_and_submit_batch(filename: str) -> Any:
    """Upload a JSONL file and create an OpenAI batch job."""
    print(f"[INFO] Uploading {filename}...")
    uploaded = client.files.create(
        file=open(filename, "rb"),
        purpose="batch"
    )
    print(f"[INFO] File uploaded: {uploaded.id}")

    print(f"[INFO] Creating batch job...")
    batch_job = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": "qa-evaluation"}
    )
    print(f"[INFO] Batch created: {batch_job.id} (status: {batch_job.status})")
    return batch_job


def wait_for_job_completion(batch_id: str, poll_interval: int = 10) -> Any:
    """Poll for batch job completion with progress updates."""
    print(f"[INFO] Waiting for batch {batch_id}...")
    start_time = time.time()

    while True:
        batch = client.batches.retrieve(batch_id)
        status = batch.status
        completed = batch.request_counts.completed if batch.request_counts else 0
        total = batch.request_counts.total if batch.request_counts else 0
        failed = batch.request_counts.failed if batch.request_counts else 0

        elapsed = (time.time() - start_time) / 60
        # print(f"  [{elapsed:.1f}m] Status: {status} | Completed: {completed}/{total} | Failed: {failed}", end="\r")

        if status in ("completed", "failed", "cancelled", "expired"):
            print()  # newline after carriage return
            break
        time.sleep(poll_interval)

    if status == "completed":
        print(f"[SUCCESS] Batch {batch_id} completed in {elapsed:.1f} min")
    else:
        print(f"[ERROR] Batch {batch_id} ended with status: {status} after {elapsed:.1f} min")
    return batch


def parse_batch_results(
    records_list: List[List[Dict]],
    parser: Optional[Callable[[str], Any]] = None
) -> tuple[Dict, List[Dict]]:
    """Parse OpenAI batch output records with optional custom parser."""
    results = {}
    errors = []

    for records in records_list:
        for rec in records:
            try:
                key = rec.get("custom_id", "")
                # OpenAI batch response structure
                response_body = rec["response"]["body"]
                result_text = response_body["choices"][0]["message"]["content"]
                if parser:
                    results[key] = parser(result_text)
                else:
                    results[key] = result_text.strip()
            except Exception as e:
                errors.append({"record": rec, "error": str(e)})

    print(f"[INFO] Successfully parsed: {len(results)} results")
    print(f"[INFO] Errors: {len(errors)}")
    return results, errors


def process_single_batch(
    batch_id: str,
    parser: Optional[Callable[[str], Any]] = None
) -> tuple[Dict, List[Dict]]:
    """Download, parse, and return results from a single completed OpenAI batch job."""
    try:
        print(f"[INFO] Downloading results for batch {batch_id}...")
        batch = client.batches.retrieve(batch_id)

        if not batch.output_file_id:
            print(f"[ERROR] No output file for batch {batch_id}")
            return {}, []

        raw = client.files.content(batch.output_file_id).text
        records = [json.loads(line) for line in raw.splitlines() if line.strip()]
        print(f"[INFO] Downloaded {len(records)} records")

        # Check for error file too
        if batch.error_file_id:
            err_raw = client.files.content(batch.error_file_id).text
            err_records = [json.loads(line) for line in err_raw.splitlines() if line.strip()]
            if err_records:
                print(f"[WARNING] {len(err_records)} errors in error file")

        results, errors = parse_batch_results([records], parser=parser)
        return results, errors

    except Exception as e:
        print(f"[ERROR] Failed to download/parse batch {batch_id}: {e}")
        return {}, []


def create_batch_files(
    inputs: List[Dict],
    batch_path_template: str,
    system_prompt: str,
    user_prompt_formatter: Callable[[Dict], str],
    response_format: Dict,
    model: str = MODEL_NAME,
    max_completion_tokens: int = 1000,
    temperature: float = 0.1,
    chunk_size: int = BATCH_CHUNK_SIZE
) -> List[str]:
    """
    Create JSONL batch files for OpenAI Batch API (/v1/chat/completions).

    Cost optimization: the system message is identical across all requests in a batch,
    so OpenAI's automatic prompt caching kicks in after the first request —
    cached input tokens are billed at 50% discount.
    """
    filenames = []
    file_idx = 1

    for i in range(0, len(inputs), chunk_size):
        batch = inputs[i:i + chunk_size]
        filename = batch_path_template.format(file_idx)
        file_idx += 1

        with open(filename, "w", encoding="utf-8") as f:
            for batch_idx, item in enumerate(batch):
                global_idx = i + batch_idx
                record = {
                    "custom_id": str(global_idx),
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": model,
                        "messages": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt_formatter(item)}
                        ],
                        "max_completion_tokens": max_completion_tokens,
                        "temperature": temperature,
                        "response_format": response_format,
                    }
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        filenames.append(filename)

    print(f"[INFO] Created {len(filenames)} batch files ({len(inputs)} total items)")
    return filenames


# ==================== INCREMENTAL SAVE/LOAD UTILITIES ====================

def save_incremental_results(batch_idx: int, results: Dict, prefix: str = "eval") -> str:
    """Save batch results incrementally to cache folder."""
    cache_path = os.path.join(CACHE_SAVE_FOLDER, f"{prefix}_results_batch_{batch_idx}.json")
    with open(cache_path, "w", encoding="utf-8") as cache_file:
        json.dump(results, cache_file, ensure_ascii=False, indent=4)
    print(f"[INFO] Saved batch {batch_idx} to: {cache_path}")
    return cache_path


def load_incremental_results(
    start_batch: int = 1,
    end_batch: Optional[int] = None,
    prefix: str = "eval"
):
    """
    Load previously saved evaluation results from cache folder.

    Args:
        start_batch: Starting batch index (default: 1)
        end_batch: Ending batch index (default: None, loads all available)
        prefix: Cache file prefix (default: "eval")

    Returns:
        tuple: (merged_results_dict, last_batch_idx)
    """
    merged_results = {}
    last_batch_idx = start_batch - 1

    batch_idx = start_batch
    while True:
        if end_batch is not None and batch_idx > end_batch:
            break

        cache_path = os.path.join(CACHE_LOAD_FOLDER, f"{prefix}_results_batch_{batch_idx}.json")

        if not os.path.exists(cache_path):
            if batch_idx == start_batch:
                print(f"[WARNING] No cache file found at batch {batch_idx}")
            break

        try:
            with open(cache_path, "r", encoding="utf-8") as cache_file:
                batch_results = json.load(cache_file)

            merged_results.update(batch_results)
            last_batch_idx = batch_idx

            print(f"[INFO] Loaded batch {batch_idx}: {len(batch_results)} results")
            batch_idx += 1

        except Exception as e:
            print(f"[ERROR] Failed to load batch {batch_idx}: {e}")
            break

    if last_batch_idx >= start_batch:
        print(f"\n[SUCCESS] Loaded batches {start_batch}-{last_batch_idx}")
        print(f"[INFO] Total merged results: {len(merged_results)}")
    else:
        print(f"[INFO] No batches loaded")

    return merged_results, last_batch_idx

# Evaluation

Evaluate generated QA pairs using a 5-criteria scoring system (0-2 points each):
- **Answerability**: Can the answer be found in the context?
- **Clarity**: Is the question clear and specific?
- **Answer Accuracy**: Is the answer correct and complete?
- **Conciseness**: Is the answer brief and to the point?
- **Usefulness**: Is the question valuable for financial QA?

## Prepare Evaluation Inputs

In [ ]:
# Prepare evaluation inputs from loaded DataFrame
# Expected input structure:
#   url: string, title: string, time: datetime,
#   qa_pairs: list[dict]

eval_inputs = []

for article_idx, row in df.iterrows():
    url = row.get('url', '')
    title = row.get('title', '')
    time_val = row.get('time', '')
    qa_pairs = row.get('qa_pairs', [])

    for q_idx, qa in enumerate(qa_pairs):
        question = qa.get('question', '') if isinstance(qa, dict) else ''
        answer = qa.get('answer', '') if isinstance(qa, dict) else ''

        eval_inputs.append({
            'eval_idx': len(eval_inputs),
            'url': url,
            'context': row.get('context', ''),
            'title': title,
            'time': str(time_val),
            'question': question,
            'answer': answer,
            'article_idx': article_idx,
            'q_idx': q_idx
        })

print(f"[INFO] Total QA pairs to evaluate: {len(eval_inputs)} from {len(df)} articles")

## Run Evaluation Batch

In [ ]:
# Define prompt formatter for evaluation (user message only — dynamic data)
def format_eval_user_prompt(item: Dict) -> str:
    return USER_PROMPT_TEMPLATE.format(
        context=item['context'],
        title=item['title'],
        timestamp=item['time'],
        question=item['question'],
        answer=item['answer']
    )

def parse_evaluation(result_text: str) -> Dict:
    eval_obj = QAEvaluationOutput.model_validate_json(result_text)
    return eval_obj.model_dump()

In [ ]:
# OpenAI Structured Outputs response_format
# Uses make_openai_strict_schema to ensure additionalProperties: false on all objects
EVAL_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "qa_evaluation",
        "strict": True,
        "schema": make_openai_strict_schema(QAEvaluationOutput)
    }
}

# Create batch files (system prompt separated for prompt caching savings)
eval_filenames = create_batch_files(
    inputs=eval_inputs,
    batch_path_template=BATCH_EVAL_PATH,
    system_prompt=SYSTEM_PROMPT,
    user_prompt_formatter=format_eval_user_prompt,
    response_format=EVAL_RESPONSE_FORMAT,
    max_completion_tokens=300,
    temperature=0.1
)

# ==================== INCREMENTAL BATCH PROCESSING ====================
if RESUME_FROM_LAST_CHECKPOINT:
    print(f"\n[INFO] Resuming from last checkpoint...")
    eval_results, last_batch = load_incremental_results(prefix="eval")
    print(f"\n[INFO] Resume point: Last completed batch = {last_batch}")
else:
    eval_results = {}
    last_batch = 0

start_batch = last_batch + 1
eval_errors = []

print(f"[INFO] Processing {len(eval_filenames)} files...")
print(f"[INFO] Will start from batch {start_batch}")

for batch_idx, filename in enumerate(eval_filenames, start=1):
    print(f"\n{'='*60}")
    print(f"[EVAL] Processing batch {batch_idx}/{len(eval_filenames)}: {filename}")
    print(f"{'='*60}")

    if batch_idx <= last_batch:
        print(f"[INFO] Skipping batch {batch_idx} (before resume point)")
        continue

    # Check if batch already processed in save folder
    cache_path = os.path.join(CACHE_SAVE_FOLDER, f"eval_results_batch_{batch_idx}.json")
    if os.path.exists(cache_path) and RESUME_FROM_LAST_CHECKPOINT:
        print(f"[INFO] Batch {batch_idx} already processed. Loading from cache...")
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                results = json.load(f)
            eval_results.update(results)
            print(f"[EVAL] Batch {batch_idx}/{len(eval_filenames)} loaded from cache")
            print(f"[EVAL] Total results so far: {len(eval_results)}")
            continue
        except Exception as e:
            print(f"[WARNING] Failed to load cache for batch {batch_idx}: {e}")
            print(f"[INFO] Reprocessing batch {batch_idx}...")

    # Step 1: Upload file and submit batch job
    job = upload_and_submit_batch(filename)

    # Step 2: Wait for completion
    completed_job = wait_for_job_completion(job.id)

    # Step 3: Download and parse results
    results, errors = process_single_batch(
        batch_id=job.id,
        parser=parse_evaluation
    )

    # Step 4: Merge results
    eval_results.update(results)
    eval_errors.extend(errors)

    # Step 5: Save incrementally
    save_incremental_results(batch_idx, results, prefix="eval")

    print(f"[EVAL] Batch {batch_idx}/{len(eval_filenames)} completed")
    print(f"[EVAL] Total results so far: {len(eval_results)}")

print(f"\n{'='*60}")
print(f"[SUCCESS] All {len(eval_filenames)} batch files processed!")
print(f"[INFO] Total results: {len(eval_results)}")
print(f"[INFO] Total errors: {len(eval_errors)}")
print(f"{'='*60}\n")

In [ ]:
# batch_ids = [
#     "batch_69ae1f80e8588190b1b636c517305672",
#     "batch_69ae214b7bdc819096b8f0efae165710",
#     "batch_69ae22215340819084b5ef1dd15d87cf"
# ]
# eval_results, eval_errors = {}, []

# for batch_id in batch_ids:
#     job = client.batches.retrieve(batch_id)
#     # Step 2: Wait for completion
#     completed_job = wait_for_job_completion(job.id)
    
#     # Step 3: Download and parse results
#     results, errors = process_single_batch(
#         batch_id=job.id,
#         parser=parse_evaluation
#     )
    
#     # Step 4: Merge results
#     eval_results.update(results)
#     eval_errors.extend(errors)

## View Sample Evaluations

In [ ]:
# Display sample evaluation outputs
import random

num_eval_samples = 3
sample_keys = random.sample(list(eval_results.keys()), min(num_eval_samples, len(eval_results)))

for i, key in enumerate(sample_keys, 1):
    eval_data = eval_results[key]
    input_data = eval_inputs[int(key)]
    
    print(f"\n{'='*80}")
    print(f"SAMPLE #{i} (key: {key})")
    print(f"{'='*80}")
    
    print(f"\nURL: {input_data['url']}")
    print(f"Title: {input_data['title']}")
    print(f"\nContext (truncated):\n   {input_data['context'][:300]}...")
    print(f"\nQuestion: {input_data['question']}")
    print(f"Answer: {input_data['answer']}")
    
    print(f"\nEVALUATION SCORES (Total: {eval_data['total_score']}/10)")
    for criterion in ['answerability', 'clarity', 'answer_accuracy', 'conciseness', 'usefulness']:
        score = eval_data[criterion]['score']
        reason = eval_data[criterion]['reason'][:200]
        print(f"   - {criterion.replace('_', ' ').title()}: {score}/2")
        print(f"     > {reason}...")
    
    print(f"\nAssessment: {eval_data['overall_assessment']}")
    print(f"Reason: {eval_data['overall_reason']}")

# Save Evaluation Results

In [ ]:
# Merge evaluation results back into the original structure
# Output: url, title, time, qa_pairs[{question, answer, eval fields...}]

# Build a lookup: (article_idx, q_idx) -> eval result
eval_lookup = {}
for idx, item in enumerate(eval_inputs):
    key = str(idx)
    eval_data = eval_results.get(key, None)
    lookup_key = (item['article_idx'], item['q_idx'])
    eval_lookup[lookup_key] = eval_data

output_rows = []
for article_idx, row in df.iterrows():
    url = row.get('url', '')
    title = row.get('title', '')
    time_val = row.get('time', '')
    context = row.get('context', '')
    qa_pairs = row.get('qa_pairs', [])

    enriched_qa_pairs = []
    for q_idx, qa in enumerate(qa_pairs):
        question = qa.get('question', '') if isinstance(qa, dict) else ''
        answer = qa.get('answer', '') if isinstance(qa, dict) else ''

        qa_entry = {
            'question': question,
            'answer': answer,
        }

        eval_data = eval_lookup.get((article_idx, q_idx), None)
        if eval_data:
            qa_entry['total_score'] = eval_data['total_score']
            qa_entry['overall_assessment'] = eval_data['overall_assessment']
            qa_entry['overall_reason'] = eval_data['overall_reason']
            for criterion in ['answerability', 'clarity', 'answer_accuracy', 'conciseness', 'usefulness']:
                qa_entry[f'{criterion}_score'] = eval_data[criterion]['score']
                qa_entry[f'{criterion}_reason'] = eval_data[criterion]['reason']
        else:
            qa_entry['total_score'] = None
            qa_entry['overall_assessment'] = 'ERROR'
            qa_entry['overall_reason'] = 'Evaluation failed'
            for criterion in ['answerability', 'clarity', 'answer_accuracy', 'conciseness', 'usefulness']:
                qa_entry[f'{criterion}_score'] = None
                qa_entry[f'{criterion}_reason'] = None

        enriched_qa_pairs.append(qa_entry)

    output_rows.append({
        'url': url,
        'title': title,
        'time': time_val,
        'context': context,
        'qa_pairs': enriched_qa_pairs
    })

df_eval = pd.DataFrame(output_rows)

# Save evaluation results (flat structure)
df_eval.to_parquet(EVAL_OUTPUT_PATH_PARQUET, engine='pyarrow', index=False)

print(f"[INFO] Saved evaluated Parquet: {EVAL_OUTPUT_PATH_PARQUET}")
print(f"[INFO] Total articles: {len(df_eval)}")
total_qa = sum(len(row['qa_pairs']) for _, row in df_eval.iterrows())
print(f"[INFO] Total QA pairs: {total_qa}")

## Evaluation Statistics

In [ ]:
# Flatten QA pairs for statistics
flat_evals = []
for _, row in df_eval.iterrows():
    for qa in row['qa_pairs']:
        flat_evals.append(qa)

df_flat = pd.DataFrame(flat_evals)
total_evaluated = len(df_flat)
valid_evaluations = df_flat[df_flat['overall_assessment'] != 'ERROR']

print("=" * 60)
print("EVALUATION STATISTICS")
print("=" * 60)

# Overall Assessment Distribution
print(f"\n[Overall Assessment Distribution]")
assessment_counts = df_flat['overall_assessment'].value_counts()
for assessment, count in assessment_counts.items():
    pct = (count / total_evaluated) * 100
    print(f"  {assessment}: {count} ({pct:.1f}%)")

# Score Statistics
print(f"\n[Score Statistics]")
valid_scores = valid_evaluations['total_score'].dropna()
if len(valid_scores) > 0:
    print(f"  Mean Score: {valid_scores.mean():.2f} / 10")
    print(f"  Median Score: {valid_scores.median():.1f} / 10")
    print(f"  Min Score: {valid_scores.min():.0f} / 10")
    print(f"  Max Score: {valid_scores.max():.0f} / 10")
    print(f"  Std Dev: {valid_scores.std():.2f}")

# Score Distribution
print(f"\n[Score Distribution]")
score_bins = [(0, 4, "Low (0-4)"), (5, 7, "Medium (5-7)"), (8, 10, "High (8-10)")]
for low, high, label in score_bins:
    count = len(valid_scores[(valid_scores >= low) & (valid_scores <= high)])
    pct = (count / len(valid_scores)) * 100 if len(valid_scores) > 0 else 0
    print(f"  {label}: {count} ({pct:.1f}%)")

# Per-Criterion Average Scores
print(f"\n[Average Scores by Criterion]")
criteria = ['answerability', 'clarity', 'answer_accuracy', 'conciseness', 'usefulness']
for criterion in criteria:
    col = f'{criterion}_score'
    if col in valid_evaluations.columns:
        avg = valid_evaluations[col].dropna().mean()
        print(f"  {criterion.replace('_', ' ').title()}: {avg:.2f} / 2")

# Summary
print(f"\n[Summary]")
print(f"  Total QA pairs evaluated: {total_evaluated}")
print(f"  Successful evaluations: {len(valid_evaluations)}")
print(f"  Failed evaluations: {total_evaluated - len(valid_evaluations)}")

accept_count = assessment_counts.get('ACCEPT', 0)
accept_pct = (accept_count / total_evaluated) * 100
print(f"  Acceptance Rate: {accept_pct:.1f}%")

print("=" * 60)